# GRPO Fine-Tuning on the AWS RL Environment

**Curriculum-driven GRPO**, built on top of the existing SFT LoRA adapter ([`Sizzing/aws-rl-sft-qwen25coder3b-adapter`](https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter)). The priority-queue curriculum in [`server/services/curriculum.py`](https://github.com/UdayKiranPadhy/aws-rl-env/blob/master/server/services/curriculum.py) picks each training task based on novelty, weakness, and spaced repetition; TRL's `GRPOTrainer` owns the generate/score/update loop; rewards flow back into the curriculum through the reward function to close the mastery loop.

Hyperparameter-tuned with Optuna, logged to Weights & Biases, checkpointed for safe resume, and published to a **separate** HuggingFace Hub repo so both the SFT and GRPO adapters coexist side-by-side.

### Architecture
```
Hosted env server (Docker, AWS_RL_ENV_POOL_SIZE=8)
        ▲ HTTPS tunnel
        │                                Colab / Kaggle VM (T4)
        │                                 └─ main python
        │  8-way httpx pool (rewards)         ├─ Unsloth: base + SFT adapter (trainable)
        ├────────────────────────       │
        │                                      ├─ TRL GRPOTrainer
        │  8-way GrpoPool WS (eval)            │    ├─ train_ds = stream from Curriculum
        └────────────────────────       │    ├─ G generations / prompt
                                               │    ├─ reward_fn:
                                               │    │    a) score via /reset + /step
                                               │    │    b) curriculum.record_result(...)
                                               │    └─ group-advantage + PPO-clip on LoRA
                                               ├─ Optuna (sqlite, resumable)
                                               └─ push → Sizzing/aws-rl-grpo-qwen25coder3b-adapter
```

### Why curriculum-driven GRPO?
Plain dataset iteration is agnostic to the model’s current weaknesses. The curriculum surfaces novel + weak tasks more often, spaced-repeats mastered ones, and promotes tier only when the agent demonstrates sustained competence. GRPO's group-relative advantage + the curriculum's per-task mastery signal compose: a group of G completions on a hard task yields a high-variance advantage signal exactly when it matters.

### Headline metrics (reported in the eval cells)
| Axis                 | SFT baseline | GRPO target | Stretch |
|----------------------|-------------:|------------:|--------:|
| `env_reward_mean`    |        ~0.85 |        ≥0.92 |   ≥0.97 |
| `env_success_rate`   |         ~85% |         ≥92% |    ≥97% |
| `recovery_rate`      |         ~40% |         ≥60% |    ≥75% |
| `drift_repair_rate`  |         ~30% |         ≥55% |    ≥70% |
| `hints_per_solved`   |         ~0.8 |        ≤0.5 |    ≤0.2 |
| `steps_to_solve`     |           ~6 |          ≤5 |      ≤4 |

## 1 · Install dependencies

Mirrors the SFT notebook's pinning strategy. Key constraints: `trl>=0.16` for `GRPOTrainer`, `transformers>=4.50,<5.0` for Unsloth compatibility (the SFT notebook ran into this too), `--force-reinstall --no-deps` on `transformers` so TRL doesn't pull an incompatible version.

In [1]:
%pip install -q --upgrade pip
%pip install -q "unsloth"
%pip install -q "trl>=0.18.2,<=0.24.0,!=0.19.0" "peft" "accelerate" "datasets" "bitsandbytes"
%pip install -q "transformers>=4.50,<5.0" --force-reinstall --no-deps
%pip install -q "optuna" "wandb" "plotly" "kaleido"
%pip install -q "httpx" "websockets" "pyyaml" "python-dotenv"
%pip install -q "huggingface_hub>=0.34,<1.0"

# openenv-core provides the `openenv` package that models.py imports from.
# The aws-rl-env pyproject.toml pins `openenv-core[core]>=0.2.2` — we match that.
# Fall back to installing directly from Meta's GitHub repo if PyPI doesn't resolve.
import subprocess, sys
def _pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode

if _pip("openenv-core[core]>=0.2.2") != 0:
    print("PyPI install of openenv-core failed; falling back to GitHub source.")
    _pip("git+https://github.com/meta-pytorch/OpenEnv.git#egg=openenv-core")

# Fail loudly here if openenv still isn't importable — better than a confusing
# ModuleNotFoundError three cells later inside models.py.
import importlib
importlib.invalidate_caches()
try:
    import openenv.core.env_server.types  # noqa: F401
    print("openenv-core installed and importable.")
except ModuleNotFoundError as e:
    raise RuntimeError(
        "openenv could not be imported after pip install. "
        "Try `%pip install git+https://github.com/meta-pytorch/OpenEnv.git` manually "
        f"and restart the runtime. Original error: {e}"
    )


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.3 MB/s eta 0:00:00
openenv-core installed and importable.


## 2 · Clone the AWS RL env repo

We need the `client.AwsRlEnv` class, `models.py` pydantic types, `scripts/grpo_pool.py`, the curriculum loader, and the task YAMLs — these power the multi-step evaluator. We do **not** install or run any env server locally; the env is hosted elsewhere.

In [2]:
from __future__ import annotations
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/UdayKiranPadhy/aws-rl-env.git"

# Detect host in priority order: Kaggle first (it often has /content too),
# then Colab, then fall back to CWD for local runs.
if Path("/kaggle/working").exists():
    REPO_DIR = Path("/kaggle/working/aws-rl-env")
elif Path("/content").exists():
    REPO_DIR = Path("/content/aws-rl-env")
else:
    REPO_DIR = (Path.cwd() / "aws-rl-env").resolve()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# If a prior partial clone left no models.py, redo it.
if not (REPO_DIR / "models.py").exists():
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

assert (REPO_DIR / "models.py").exists(), f"models.py missing under {REPO_DIR}"
print(f"Repo at {REPO_DIR} — ready. sys.path[0]={sys.path[0]}")


Repo at /content/aws-rl-env — ready. sys.path[0]=/content/aws-rl-env


## 3 · Runtime detection

Matches the SFT notebook so we pick `fp16` on T4 (bf16 unsupported) and `bf16` on A100/H100. Also sets the PyTorch allocator env var that cut OOMs on long Unsloth runs.

In [4]:
from dataclasses import dataclass
import torch

IS_COLAB  = True
IS_KAGGLE = False

OUT_DIR = Path("/content/out") if IS_COLAB else Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")


@dataclass(frozen=True)
class Runtime:
    gpu_name: str
    use_fp16: bool
    use_bf16: bool


def detect_runtime() -> Runtime:
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. This notebook needs CUDA (T4/A100/H100).")
    name = torch.cuda.get_device_name(0)
    is_t4 = "T4" in name
    return Runtime(gpu_name=name, use_fp16=is_t4, use_bf16=not is_t4)


RT = detect_runtime()
print(f"GPU: {RT.gpu_name}  |  fp16={RT.use_fp16}  bf16={RT.use_bf16}  |  OUT_DIR={OUT_DIR}")

GPU: Tesla T4  |  fp16=True  bf16=False  |  OUT_DIR=/content/out


## 4 · Training configuration

One frozen `TrainingConfig` dataclass holds every tunable hyperparameter. Optuna hands trials a mutated copy of this same dataclass, so the trial path and the final-run path go through identical code.

Separate `ModelSpec` and `PathsSpec` keep non-tunable identifiers out of the search space.

In [5]:
from dataclasses import replace


@dataclass(frozen=True)
class ModelSpec:
    base_model:   str = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
    sft_adapter:  str = "Sizzing/aws-rl-sft-qwen25coder3b-adapter"
    grpo_adapter: str = "Sizzing/aws-rl-grpo-qwen25coder3b-adapter"   # NEW repo — SFT repo untouched
    dataset_repo: str = "Sizzing/aws-rl-sft"
    max_seq_length: int = 512


@dataclass(frozen=True)
class TrainingConfig:
    # GRPO knobs (Optuna-tunable)
    learning_rate:   float = 5e-6
    beta:            float = 0.04    # KL coef vs. reference (frozen SFT adapter)
    num_generations: int   = 8       # G in GRPO
    temperature:     float = 0.9
    top_p:           float = 0.95
    lora_r:          int   = 16
    lora_alpha_mul:  int   = 2
    # Empirical defaults — hold fixed
    max_prompt_length:     int = 384
    max_completion_length: int = 128
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs:      int = 1     # ignored for IterableDataset; max_steps drives termination
    save_steps:            int = 25
    save_total_limit:      int = 3
    eval_steps:            int = 50
    warmup_ratio:          float = 0.05
    seed:                  int = 42


@dataclass(frozen=True)
class PipelineConfig:
    env_pool_size: int = 8
    n_trials:      int = 6
    trial_max_steps:  int = 30
    final_max_steps:  int = 200   # curriculum-driven; bounds the infinite IterableDataset stream
    val_subset_size:  int = 20
    eval_reserve_cap: int = 100
    wandb_project: str = "AWS-RL-GRPO"
    wandb_entity:  str = "sizzing-sizzing"


MODEL = ModelSpec()
TRAIN = TrainingConfig()
PIPE  = PipelineConfig()

print("ModelSpec:",   MODEL)
print("TrainingConfig defaults:", TRAIN)
print("PipelineConfig:", PIPE)

ModelSpec: ModelSpec(base_model='unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit', sft_adapter='Sizzing/aws-rl-sft-qwen25coder3b-adapter', grpo_adapter='Sizzing/aws-rl-grpo-qwen25coder3b-adapter', dataset_repo='Sizzing/aws-rl-sft', max_seq_length=512)
TrainingConfig defaults: TrainingConfig(learning_rate=5e-06, beta=0.04, num_generations=8, temperature=0.9, top_p=0.95, lora_r=16, lora_alpha_mul=2, max_prompt_length=384, max_completion_length=128, per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=1, save_steps=25, save_total_limit=3, eval_steps=50, warmup_ratio=0.05, seed=42)
PipelineConfig: PipelineConfig(env_pool_size=8, n_trials=6, trial_max_steps=30, final_max_steps=200, val_subset_size=20, eval_reserve_cap=100, wandb_project='AWS-RL-GRPO', wandb_entity='sizzing-sizzing')


## 5 · Authenticate: HF Hub, Weights & Biases, env URL

Three secrets are required:

| Secret                  | Scope         | Used for                                 |
|-------------------------|---------------|------------------------------------------|
| `HF_TOKEN`              | read + write  | pull SFT adapter, push GRPO adapter      |
| `WANDB_API_KEY`         | default       | project logging                          |
| `AWS_RL_ENV_BASE_URL`   | —              | URL of the hosted env (cloudflared/ngrok) |

We also dummy-push a tiny file to verify the HF token has write access to the `Sizzing` org **before** spending hours training.

In [ ]:
def _load_secret(name: str) -> str:
    """Read a secret from Colab userdata, Kaggle UserSecrets, or the env."""
    if IS_COLAB:
        from google.colab import userdata
        try: return userdata.get(name)
        except Exception: pass
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        try: return UserSecretsClient().get_secret(name)
        except Exception: pass
    val = os.environ.get(name)
    if not val:
        raise RuntimeError(f"Secret {name!r} is missing. Set it in Colab/Kaggle secrets or as an env var.")
    return val


HF_TOKEN         = _load_secret("HF_TOKEN")
WANDB_API_KEY    = _load_secret("WANDB_API_KEY")
ENV_BASE_URL     = _load_secret("AWS_RL_ENV_BASE_URL").rstrip("/")

# Keep the tunnel URL out of wandb.config to avoid leaking it on the public dashboard
os.environ["WANDB_DISABLE_GIT"] = "true"

from huggingface_hub import login as hf_login, HfApi
import wandb

hf_login(token=HF_TOKEN, add_to_git_credential=False)
wandb.login(key=WANDB_API_KEY)


def verify_hub_write_scope(repo_id: str) -> None:
    """Ensure the HF token can create repos under the target org before training.

    Without this, we'd discover permission failures *after* a multi-hour run.
    """
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=repo_id, private=True, exist_ok=True, repo_type="model")
    probe = OUT_DIR / ".hub_write_probe.txt"
    probe.write_text("ok")
    api.upload_file(path_or_fileobj=str(probe), path_in_repo=".hub_write_probe.txt",
                    repo_id=repo_id, commit_message="probe: verify write scope")
    probe.unlink()


verify_hub_write_scope(MODEL.grpo_adapter)
print("All secrets loaded; HF write access to", MODEL.grpo_adapter, "verified.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sizzing (sizzing-sizzing) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
No files have been modified since last commit. Skipping to prevent empty commit.


All secrets loaded; HF write access to Sizzing/aws-rl-grpo-qwen25coder3b-adapter verified.


## 6 · Connect to the hosted env + health check

The env server runs **outside** this VM. We only assert reachability and that its internal pool has 8 slots (required for parallel rollouts). If either fails, we stop **before** loading the model.

In [7]:
import httpx


def probe_env_http(base_url: str) -> dict:
    """Cheap reachability check: GET /schema. Raises on HTTP error.

    Does NOT try to validate pool size from /state — AwsRlState doesn't
    expose it. Pool capacity is verified later via a real GrpoPool
    connection attempt (§6b) which is the only honest way to check.
    """
    with httpx.Client(base_url=base_url, timeout=10.0) as c:
        schema = c.get("/schema").raise_for_status().json()
    return {"schema": schema}


probe = probe_env_http(ENV_BASE_URL)
print(f"Env reachable. Action schema keys: {list(probe['schema'].keys())}")
print(f"Note: POOL_SIZE={PIPE.env_pool_size} is assumed — set by the operator "
      f"when launching the server with AWS_RL_ENV_POOL_SIZE=8. Capacity is "
      f"verified by the GrpoPool connection attempt below.")

Env reachable. Action schema keys: ['action', 'observation', 'state']
Note: POOL_SIZE=8 is assumed — set by the operator when launching the server with AWS_RL_ENV_POOL_SIZE=8. Capacity is verified by the GrpoPool connection attempt below.


## 7 · Curriculum-driven prompt stream + fixed val / reserve sets

Instead of feeding a static dataset, we stream prompts from the repo's [`Curriculum`](https://github.com/UdayKiranPadhy/aws-rl-env/blob/master/server/services/curriculum.py) — the same priority-queue curriculum the env already implements:

- **novelty bonus** for untried tasks (+100)
- **weakness weighting** `(1 − recent_success_rate) × 50` per task
- **spaced repetition** on graduated tasks at 3→66→12→24→48 episode intervals
- **recency penalty** to avoid drilling the same task back-to-back
- **tier promotion** with fast-track when success rate crosses threshold

TRL's `GRPOTrainer` accepts a `datasets.IterableDataset`; we wrap `curriculum.next_task()` in a generator and feed it in. The feedback loop closes inside the reward function (next cell): after scoring a group of G completions for a task, it calls `curriculum.record_result(task, achieved, mean_reward)`, which updates mastery, promotes tiers, and re-ranks the priority queue for the next step.

`VAL_DS` (fixed 20-row subset from the HF dataset's `validation` split) and `RESERVE_DS` (up to 100 rows from `reserve`) stay fixed for comparable before/after evaluation.

In [ ]:
from datasets import load_dataset, Dataset
from models import Task, TaskDifficulty
from server.services.curriculum import Curriculum, load_tier
import yaml


def build_task_map(tasks_dir: Path) -> dict[int, Task]:
    """Flatten every task YAML into a dict keyed by task_id.

    The reward function only has task_id to work with; this map lets it
    recover the full Task object needed to serialise over HTTP to /reset.
    """
    m: dict[int, Task] = {}
    for tier in TaskDifficulty:
        try:
            for t in load_tier(tier, tasks_dir):
                m[int(t.task_id)] = t
        except FileNotFoundError:
            continue
    return m


def load_drift_task_ids(tasks_dir: Path) -> set[int]:
    """Drift tasks live in drift.yaml and get folded into the EXPERT tier by
    the curriculum loader. We scan the file directly so we can still identify
    them for drift_repair_rate in multi-step eval.
    """
    ids: set[int] = set()
    drift_path = tasks_dir / "drift.yaml"
    if drift_path.exists():
        with open(drift_path) as f:
            for entry in (yaml.safe_load(f) or []):
                ids.add(int(entry["task_id"]))
    return ids


SYSTEM_PROMPT = (
    "You are an expert AWS SRE agent. You operate a simulated AWS cloud by "
    "emitting one AWS CLI command at a time. You will see the task description, "
    "then reply with EXACTLY ONE AWS CLI command on a single line starting with "
    "\"aws \". No explanation, no markdown, no quotes — just the command."
)


def task_to_row(task: Task) -> dict:
    """Convert a Task into the (prompt, task_id) schema GRPOTrainer expects."""
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"TASK: {task.description}"},
        ],
        "task_id": int(task.task_id),
    }


def make_curriculum_dataset(curriculum: Curriculum, n_rows: int) -> Dataset:
    """Pre-materialise N curriculum-picked prompts as a finite `datasets.Dataset`.

    TRL's GRPOTrainer explicitly rejects IterableDataset (see TRL issue #3213:
    ``NotImplementedError: Iterable datasets are not yet supported``). That
    kills the original on-demand `curriculum.next_task()` streaming design —
    the trainer must see a Dataset with a known `__len__`.

    Compromise: sample `n_rows` prompts up front. The curriculum's novelty +
    weakness + recency heuristics still apply *across the draw* (every
    `next_task()` pops the current top-of-queue and ages neighbouring tasks
    via recency), so we get a sensibly ordered warm-start sample — not
    uniform-random. What we lose is live re-ranking between steps: mastery
    updates made by the reward function during training don't feed back
    into selection until a fresh dataset is built. `curriculum.record_result`
    still runs inside the reward function so mastery metrics remain accurate
    for wandb logging and end-of-run stats.

    Size `n_rows` to `max_steps * per_device_train_batch_size *
    gradient_accumulation_steps`, plus a buffer so `num_train_epochs=1`
    never exhausts the dataset before `max_steps` terminates training.
    """
    return Dataset.from_list(
        [task_to_row(curriculum.next_task()) for _ in range(n_rows)]
    )


def build_val_dataset(dataset_repo: str, task_map: dict[int, Task],
                      val_size: int, seed: int = 42) -> Dataset:
    """Fixed validation subset for comparable before/after eval."""
    raw = load_dataset(dataset_repo, token=HF_TOKEN)
    val_single = raw["validation"].filter(
        lambda r: r["step_idx"] == 0 and int(r["task_id"]) in task_map
    )
    val = val_single.shuffle(seed=seed).select(
        range(min(val_size, len(val_single)))
    )
    return val.map(
        lambda r: {"prompt": r["messages"][:2], "task_id": int(r["task_id"])},
        remove_columns=[c for c in val.column_names if c not in ("prompt", "task_id")],
    )


def build_reserve_dataset(dataset_repo: str,
                          task_map: dict[int, Task]) -> Dataset | None:
    """Reserve split for the multi-step eval."""
    raw = load_dataset(dataset_repo, token=HF_TOKEN)
    if "reserve" not in raw:
        return None
    reserve_single = raw["reserve"].filter(
        lambda r: r["step_idx"] == 0 and int(r["task_id"]) in task_map
    )
    return reserve_single.map(
        lambda r: {"prompt": r["messages"][:2], "task_id": int(r["task_id"])},
        remove_columns=[c for c in reserve_single.column_names
                         if c not in ("prompt", "task_id")],
    )


_tasks_dir = REPO_DIR / "server" / "services" / "tasks"
TASK_MAP = build_task_map(_tasks_dir)
DRIFT_TASK_IDS = load_drift_task_ids(_tasks_dir)
VAL_DS = build_val_dataset(MODEL.dataset_repo, TASK_MAP,
                            PIPE.val_subset_size, TRAIN.seed)
RESERVE_DS = build_reserve_dataset(MODEL.dataset_repo, TASK_MAP)

print(f"TASK_MAP:      {len(TASK_MAP)} tasks across "
      f"{len({t.difficulty for t in TASK_MAP.values()})} tiers")
print(f"DRIFT_TASK_IDS: {len(DRIFT_TASK_IDS)} drift tasks")
print(f"VAL_DS:        {len(VAL_DS)} rows (fixed, for before/after comparison)")
print(f"RESERVE_DS:    {len(RESERVE_DS) if RESERVE_DS else 0} rows (multi-step eval)")


## 8 · Reward functions + curriculum feedback

Three reward functions passed to `GRPOTrainer.reward_funcs`:

| Reward           | Weight | Signal                                                     |
|------------------|-------:|------------------------------------------------------------|
| `env_reward`     |   1.0  | Real env reward from `/reset` + `/step` against the task   |
| `format_reward`  |  0.15  | 1.0 if completion starts with `aws `, else 0.0             |
| `length_reward`  |  0.05  | Soft length prior: 1.0 ≤120 chars, decays to 0.0 by 400    |

`env_reward` also closes the curriculum loop. TRL emits the batch as `batch_size × num_generations` flattened completions with `task_id` repeated G times per prompt. We group by task_id, and for each group call `curriculum.record_result(task, achieved=any(r≥1.0), reward=mean)`. That one line is the bridge: TRL owns iteration, the curriculum owns task selection, and the reward function owns the feedback edge. No custom training loop; all the quality-of-life of TRL (wandb, checkpoint, resume, Optuna) is preserved.

Thread safety: env HTTP calls run on the pool threads, but reward aggregation + `record_result` both happen on the main thread after the pool join, so no locking is needed.

In [ ]:
import asyncio
import threading
import traceback
from collections import defaultdict
from typing import Callable, Optional

from client import AwsRlEnv
from models import AwsRlAction
from websockets.exceptions import ConnectionClosed


def extract_aws_command(raw: str) -> str:
    """Pure: first `aws ...` line from model output; fall back to `aws help`.

    Pure (no I/O) so it can be unit-tested and reused by the multi-step
    evaluator. `aws help` is a safe fallback: always succeeds in the env,
    earns 0 task reward but never crashes a batch.
    """
    for line in raw.splitlines():
        line = line.strip().strip("`").strip()
        if line.startswith("aws "):
            return line
    return "aws help"


class EnvRewardClient:
    """Persistent-WebSocket reward client against a pooled env server.

    Why WebSocket and not HTTP /reset + /step: under `AWS_RL_ENV_POOL_SIZE>1`
    the server's HTTP path uses an env *factory* — every request builds a
    fresh `AwsRlEnvironment` from the pool factory, so `/step` on request 2
    lands on a different env than `/reset` on request 1 and trips
    ``assert self._episode is not None, "Call reset() before step()"``.
    Only WebSocket sessions hold a MiniStack slot across calls.

    Design:
      - Dedicated background thread owns an asyncio loop.
      - On startup, N AwsRlEnv WebSocket sessions connect in parallel and
        sit in an asyncio.Queue acting as a free-list.
      - score_batch() is synchronous (TRL calls reward funcs sync): it
        submits one async task per (task, command) pair to the loop via
        run_coroutine_threadsafe, each task acquires a free session from
        the queue, does reset+step, returns the env to the queue.
      - Reconnect-on-failure: Cloudflare tunnels can idle-close WS
        sessions, but the client keeps a reference to the (now-closed)
        socket so OpenEnv's _ensure_connected() is a no-op. We catch
        ConnectionClosed explicitly, call disconnect(), reconnect, and
        retry once before giving up.

    Counters (success / timeout / conn_err / reconnect) let the trainer
    log env health without inspecting internal state. `last_error`
    captures the most recent exception. `verbose_errors=True` prints
    full tracebacks per failure (noisy — only enable while debugging).
    """

    def __init__(
        self,
        base_url: str,
        pool_size: int = 8,
        timeout_s: float = 60.0,
        verbose_errors: bool = False,
    ):
        self.base_url = base_url
        self.pool_size = pool_size
        self.timeout_s = timeout_s
        self.verbose_errors = verbose_errors
        self.success = 0
        self.timeout = 0
        self.conn_err = 0
        self.reconnect = 0
        self.last_error: Optional[str] = None
        self._loop: Optional[asyncio.AbstractEventLoop] = None
        self._thread: Optional[threading.Thread] = None
        self._queue: Optional[asyncio.Queue] = None
        self._envs: list = []
        self._ready = threading.Event()
        self._setup_error: Optional[BaseException] = None
        self._start()

    def _start(self) -> None:
        def run():
            loop = asyncio.new_event_loop()
            self._loop = loop
            asyncio.set_event_loop(loop)
            try:
                loop.run_until_complete(self._setup())
            except BaseException as e:
                self._setup_error = e
                self._ready.set()
                return
            self._ready.set()
            loop.run_forever()

        self._thread = threading.Thread(target=run, daemon=True, name="env-reward")
        self._thread.start()
        self._ready.wait()
        if self._setup_error is not None:
            raise RuntimeError(
                f"EnvRewardClient failed to connect {self.pool_size} WS sessions "
                f"to {self.base_url}: {self._setup_error!r}"
            )

    async def _setup(self) -> None:
        self._queue = asyncio.Queue()
        self._envs = [AwsRlEnv(base_url=self.base_url) for _ in range(self.pool_size)]
        try:
            await asyncio.gather(*(e.connect() for e in self._envs))
        except BaseException:
            await asyncio.gather(
                *(e.close() for e in self._envs), return_exceptions=True
            )
            raise
        for e in self._envs:
            self._queue.put_nowait(e)

    async def _reconnect(self, env) -> None:
        """Discard the dead socket and open a fresh WS session in-place."""
        self.reconnect += 1
        try:
            await env.disconnect()
        except Exception:
            pass
        await env.connect()

    async def _reset_and_step(self, env, task: Task, command: str) -> float:
        await asyncio.wait_for(env.reset(task=task), timeout=self.timeout_s)
        res = await asyncio.wait_for(
            env.step(AwsRlAction(command=command)), timeout=self.timeout_s
        )
        return float(res.reward)

    async def _score_one(self, task: Task, command: str) -> float:
        env = await self._queue.get()
        try:
            try:
                reward = await self._reset_and_step(env, task, command)
                self.success += 1
                return reward
            except ConnectionClosed as e:
                # Cloudflare / server idle-closed the socket. Reconnect and retry.
                self.last_error = f"reconnect after {type(e).__name__}: {e}"
                if self.verbose_errors:
                    print(f"[reward] {self.last_error} — reconnecting")
                await self._reconnect(env)
                reward = await self._reset_and_step(env, task, command)
                self.success += 1
                return reward
        except asyncio.TimeoutError:
            self.timeout += 1
            self.last_error = "asyncio.TimeoutError"
            if self.verbose_errors:
                traceback.print_exc()
            return 0.0
        except Exception as e:
            self.conn_err += 1
            self.last_error = f"{type(e).__name__}: {e}"
            if self.verbose_errors:
                traceback.print_exc()
            return 0.0
        finally:
            self._queue.put_nowait(env)

    async def _score_batch_async(
        self, tasks: list, commands: list[str]
    ) -> list[float]:
        return list(
            await asyncio.gather(
                *(self._score_one(t, c) for t, c in zip(tasks, commands))
            )
        )

    def score_batch(self, tasks: list, commands: list[str]) -> list[float]:
        """Parallel scoring; preserves input order."""
        assert self._loop is not None
        fut = asyncio.run_coroutine_threadsafe(
            self._score_batch_async(tasks, commands), self._loop
        )
        return fut.result()

    def drain_counters(self) -> dict:
        out = {
            "success": self.success,
            "timeout": self.timeout,
            "conn_err": self.conn_err,
            "reconnect": self.reconnect,
            "last_error": self.last_error,
        }
        self.success = self.timeout = self.conn_err = self.reconnect = 0
        self.last_error = None
        return out

    def close(self) -> None:
        if self._loop is None or not self._loop.is_running():
            return

        async def _close():
            await asyncio.gather(
                *(e.disconnect() for e in self._envs), return_exceptions=True
            )

        try:
            asyncio.run_coroutine_threadsafe(_close(), self._loop).result(timeout=30)
        except Exception:
            pass
        self._loop.call_soon_threadsafe(self._loop.stop)
        if self._thread is not None:
            self._thread.join(timeout=5)


def build_reward_funcs(env: EnvRewardClient,
                       task_map: dict[int, Task],
                       curriculum: Optional[Curriculum] = None
                       ) -> tuple[Callable, Callable, Callable]:
    """Return (env_reward, format_reward, length_reward).

    When `curriculum` is provided, env_reward calls curriculum.record_result()
    once per unique task_id in the batch — one record = one group = one
    curriculum episode. This is what makes the training loop curriculum-driven
    even while TRL owns the outer iteration.
    """
    def _text_of(c) -> str:
        return c if isinstance(c, str) else c[0]["content"]

    def env_reward(prompts, completions, task_id=None, **kw):
        tids = task_id if task_id is not None else kw["task_id"]
        tasks = [task_map[int(t)] for t in tids]
        cmds  = [extract_aws_command(_text_of(c)) for c in completions]
        rewards = env.score_batch(tasks, cmds)

        # Group by task_id and feed each group back to the curriculum. TRL emits
        # G completions per prompt consecutively (all sharing one task_id), so
        # grouping recovers the GRPO semantics cleanly: one record per prompt,
        # achieved iff any rollout hit reward>=1.0, recorded reward = group mean.
        if curriculum is not None:
            by_tid: dict[int, list[float]] = defaultdict(list)
            for tid, r in zip(tids, rewards):
                by_tid[int(tid)].append(r)
            for tid, group in by_tid.items():
                curriculum.record_result(
                    task_map[tid],
                    achieved=any(r >= 1.0 for r in group),
                    reward=sum(group) / len(group),
                )
        return rewards

    def format_reward(prompts, completions, **kw):
        return [1.0 if _text_of(c).lstrip().startswith("aws ") else 0.0
                for c in completions]

    def length_reward(prompts, completions, **kw):
        out = []
        for c in completions:
            n = len(_text_of(c))
            out.append(1.0 if n <= 120 else max(0.0, 1.0 - (n - 120) / 280.0))
        return out

    return env_reward, format_reward, length_reward


# Re-run protection: if this cell has been run before, the old ENV_CLIENT's
# background thread is still holding 8 WebSocket sessions. Close it cleanly
# (sends {"type":"close"} so the server's /ws handler reaches its finally
# block and calls _destroy_session → releases the MiniStack slot). Without
# this, every re-run compounds the leak until the server hits 8/8 capacity.
try:
    _prev = ENV_CLIENT  # noqa: F821
except NameError:
    pass
else:
    print("Closing previous ENV_CLIENT to release its WS sessions...")
    try:
        _prev.close()
    except Exception as _e:
        print(f"  (ignored error during previous close: {_e!r})")

# verbose_errors=True for the first run so the smoke test surfaces any WS /
# reset / step exception with a full traceback. Flip off after it passes.
ENV_CLIENT = EnvRewardClient(
    base_url=ENV_BASE_URL,
    pool_size=PIPE.env_pool_size,
    verbose_errors=True,
)

# Smoke test uses a non-curriculum client to avoid polluting any curriculum state.
# Task 0 ("List all S3 buckets") matches the `aws s3 ls` completion.
_smoke_task = TASK_MAP[0]
_smoke_env, _smoke_fmt, _smoke_len = build_reward_funcs(ENV_CLIENT, TASK_MAP, curriculum=None)
_smoke = _smoke_env(
    prompts=[None] * 2,
    completions=["aws s3 ls", "aws s3 ls"],
    task_id=[_smoke_task.task_id, _smoke_task.task_id],
)
print(f"Reward smoke test on task {_smoke_task.task_id}: {_smoke}")
print(f"Env counters: {ENV_CLIENT.drain_counters()}")
assert min(_smoke) > 0.5, "Reward smoke test failed — env or reward wiring broken"


## 9 · Load the SFT adapter as the starting policy

We go through `PeftModel.from_pretrained(base, sft_adapter, is_trainable=True)` explicitly (rather than `FastLanguageModel.from_pretrained(adapter_repo)`) so there is no ambiguity about the adapter landing in trainable mode — GRPO must be able to update the LoRA weights.

This helper is also used later by the final-run and Optuna-trial paths, so it lives in its own cell.

In [9]:
import gc


def load_policy(spec: ModelSpec, trainable: bool = True):
    """Load base (4-bit) + SFT adapter.

    `trainable=True` gives a PeftModel with is_trainable=True — the correct
    mode for GRPO. `trainable=False` loads the adapter for inference only
    and invokes `FastLanguageModel.for_inference` for Unsloth's fused
    inference kernels (used during evaluation).
    """
    from unsloth import FastLanguageModel
    from peft import PeftModel

    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=spec.base_model,
        max_seq_length=spec.max_seq_length,
        load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(base, spec.sft_adapter, is_trainable=trainable)
    if trainable:
        FastLanguageModel.for_training(model)
        # PeftModel.from_pretrained doesn't wire up the input-require-grads
        # forward hook on the embeddings. With Unsloth's gradient
        # checkpointing + 4-bit frozen base, that breaks the autograd path
        # between the loss and the trainable LoRA adapters — backward
        # errors with "element 0 of tensors does not require grad". The
        # hook marks the embedding output as requires_grad so gradients
        # can flow into the adapters. FastLanguageModel.get_peft_model
        # does this automatically; we don't use that path because we're
        # loading an existing SFT adapter, not creating a fresh one.
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
    else:
        FastLanguageModel.for_inference(model)
    return model, tokenizer


def free_model(model) -> None:
    """Release GPU memory held by a model + its optimizer state."""
    del model
    gc.collect()
    torch.cuda.empty_cache()


# Sanity: load + confirm LoRA params are trainable.
_probe_model, _probe_tok = load_policy(MODEL, trainable=True)
_trainable = [n for n, p in _probe_model.named_parameters() if p.requires_grad]
print(f"Loaded {MODEL.sft_adapter}. Trainable params: {len(_trainable)} tensors; sample:", _trainable[:3])
assert any("lora" in n.lower() for n in _trainable), "No LoRA params marked trainable — load path is wrong"
free_model(_probe_model)
del _probe_tok
gc.collect(); torch.cuda.empty_cache()
print("Model load path verified.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.7: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

Loaded Sizzing/aws-rl-sft-qwen25coder3b-adapter. Trainable params: 288 tensors; sample: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight']
Model load path verified.


## 10 · Baseline eval — SFT adapter, single-step env reward

Single-pass eval on the val set. This is the "before" column of the headline comparison. The richer *multi-step* eval happens later; this one is only here to confirm the SFT-loaded model outputs sane commands against the env **before** we start GRPO.

Written as a small `evaluate_single_step` helper because GRPO's inner trial loop needs the same logic.

In [ ]:
import json
import statistics as _stats
from dataclasses import asdict


@dataclass
class SingleStepMetrics:
    """One row of headline comparison numbers."""
    env_reward_mean:   float
    env_success_rate:  float   # fraction with reward >= 1.0
    format_pct:        float
    n:                 int

    def as_dict(self) -> dict:
        return asdict(self)


def evaluate_single_step(model, tokenizer, dataset, env: EnvRewardClient,
                         task_map: dict[int, Task],
                         max_new_tokens: int = 128) -> SingleStepMetrics:
    """Generate one command per prompt, score against the env, summarize."""
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    formats: list[float] = []
    tasks_to_score: list[Task] = []
    cmds_to_score:  list[str]  = []

    for row in dataset:
        prompt_text = tokenizer.apply_chat_template(
            row["prompt"], tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(
            ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
        )
        formats.append(1.0 if text.lstrip().startswith("aws ") else 0.0)
        tasks_to_score.append(task_map[int(row["task_id"])])
        cmds_to_score.append(extract_aws_command(text))

    # Score all env rewards in parallel across the 8 server slots
    rewards = env.score_batch(tasks_to_score, cmds_to_score)

    FastLanguageModel.for_training(model)

    return SingleStepMetrics(
        env_reward_mean=float(_stats.mean(rewards)),
        env_success_rate=sum(r >= 1.0 for r in rewards) / len(rewards),
        format_pct=float(_stats.mean(formats)),
        n=len(rewards),
    )


# Run the SFT-only baseline and persist it alongside Optuna + checkpoints
_baseline_model, _baseline_tok = load_policy(MODEL, trainable=False)
baseline_metrics = evaluate_single_step(
    _baseline_model, _baseline_tok, VAL_DS, ENV_CLIENT, TASK_MAP,
    max_new_tokens=TRAIN.max_completion_length,
)
(OUT_DIR / "baseline_single_step.json").write_text(json.dumps(baseline_metrics.as_dict(), indent=2))
free_model(_baseline_model); del _baseline_tok
gc.collect(); torch.cuda.empty_cache()

print("SFT baseline (single-step, val):", baseline_metrics.as_dict())


In [ ]:
import optuna


def suggest_training_config(trial: optuna.Trial, defaults: TrainingConfig) -> TrainingConfig:
    """Return a mutated copy of `defaults` with Optuna-sampled hparams.

    One function, one diff — keeps the search space auditable.
    """
    return replace(
        defaults,
        learning_rate   = trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
        beta            = trial.suggest_float("beta", 0.0, 0.1),
        num_generations = trial.suggest_categorical("num_generations", [4, 8]),
        temperature     = trial.suggest_float("temperature", 0.7, 1.0),
        lora_r          = trial.suggest_categorical("lora_r", [8, 16, 32]),
        lora_alpha_mul  = trial.suggest_categorical("lora_alpha_mul", [1, 2]),
    )


def trial_objective(trial: optuna.Trial) -> float:
    """Short curriculum-driven GRPO run + val eval. Returns mean env reward on VAL_DS."""
    trial_cfg = suggest_training_config(trial, TRAIN)
    output_dir = str(OUT_DIR / f"optuna/trial-{trial.number}")

    # Fresh curriculum per trial — otherwise mastery and tier progression bleed
    # across trials, making Optuna's hparam comparison unfair.
    trial_curriculum = Curriculum(tasks_dir=_tasks_dir)
    # Pre-materialise enough prompts for the whole trial: max_steps batches,
    # each of size pdtbs × grad_accum prompts, plus a small buffer so eval
    # or off-by-ones at the end of epoch 1 don't exhaust the dataset before
    # max_steps stops training.
    trial_n_rows = int(
        PIPE.trial_max_steps
        * trial_cfg.per_device_train_batch_size
        * trial_cfg.gradient_accumulation_steps
        * 1.2
    )
    trial_train_ds = make_curriculum_dataset(trial_curriculum, trial_n_rows)
    trial_env_r, trial_fmt_r, trial_len_r = build_reward_funcs(
        ENV_CLIENT, TASK_MAP, trial_curriculum,
    )

    model, tokenizer = load_policy(MODEL, trainable=True)

    run = wandb.init(
        project=PIPE.wandb_project, entity=PIPE.wandb_entity,
        name=f"optuna-trial-{trial.number}",
        config={**asdict(trial_cfg), "trial_number": trial.number, "curriculum": True},
        reinit=True,
    )

    trainer = build_trainer(
        model, tokenizer,
        train_ds=trial_train_ds,
        eval_ds=VAL_DS,
        reward_funcs=(trial_env_r, trial_fmt_r, trial_len_r),
        cfg=trial_cfg,
        output_dir=output_dir,
        wandb_run_name=run.name,
        use_fp16=RT.use_fp16, use_bf16=RT.use_bf16,
        max_steps=PIPE.trial_max_steps,
        save_strategy="no",
    )

    try:
        trainer.train()
        metrics = evaluate_single_step(
            trainer.model, tokenizer, VAL_DS, ENV_CLIENT, TASK_MAP,
            max_new_tokens=trial_cfg.max_completion_length,
        )
        score = metrics.env_reward_mean
        wandb.log({
            "trial/env_reward_mean":  score,
            "trial/env_success_rate": metrics.env_success_rate,
            "trial/curriculum_tier":  trial_curriculum.current_difficulty.value,
            "trial/graduated_tasks":  len(trial_curriculum.get_stats().get("graduated_tasks", [])),
        })
    finally:
        wandb.finish()
        free_model(trainer); free_model(model); del tokenizer
        gc.collect(); torch.cuda.empty_cache()

    trial.report(score, step=PIPE.trial_max_steps)
    if trial.should_prune():
        raise optuna.TrialPruned()
    return score


print("Curriculum-driven Optuna objective defined.")


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback


class EnvHealthCallback(TrainerCallback):
    """Log env health counters + drain them every N steps.

    Also provides an early-warning bail-out: if every scoring call in a
    window came back as timeout/conn_err, the hosted env is probably down
    and we want to stop training rather than polluting the adapter with
    zero-reward updates.
    """

    def __init__(self, env_client: EnvRewardClient, probe_every: int = 50,
                 fail_threshold: int = 32) -> None:
        self.env = env_client
        self.probe_every = probe_every
        self.fail_threshold = fail_threshold

    def on_log(self, args, state, control, logs=None, **kw):
        if state.global_step == 0 or state.global_step % self.probe_every != 0:
            return
        counters = self.env.drain_counters()
        try:
            wandb.log({f"env/{k}": v for k, v in counters.items()})
        except Exception:
            pass
        if counters["timeout"] + counters["conn_err"] >= self.fail_threshold:
            print(
                f"[EnvHealthCallback] {counters} at step {state.global_step}. "
                "Stopping training — check the hosted env / tunnel."
            )
            control.should_training_stop = True


def build_trainer(model, tokenizer, train_ds, eval_ds,
                  reward_funcs: Iterable[Callable],
                  cfg: TrainingConfig, *,
                  output_dir: str, wandb_run_name: str,
                  use_fp16: bool, use_bf16: bool,
                  max_steps: int | None = None,
                  save_strategy: str = "steps",
                  eval_strategy: str = "steps") -> GRPOTrainer:
    """Assemble a GRPOTrainer from a typed TrainingConfig."""
    args = GRPOConfig(
        output_dir=output_dir, run_name=wandb_run_name,
        num_generations=cfg.num_generations, beta=cfg.beta,
        temperature=cfg.temperature, top_p=cfg.top_p,
        max_prompt_length=cfg.max_prompt_length,
        max_completion_length=cfg.max_completion_length,
        learning_rate=cfg.learning_rate, lr_scheduler_type="cosine",
        optim="adamw_8bit", weight_decay=0.0, max_grad_norm=1.0,
        warmup_ratio=cfg.warmup_ratio,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        # TRL's GRPOConfig asserts per_device_eval_batch_size * num_processes is
        # divisible by num_generations (one eval prompt produces G completions;
        # anything smaller can't form a group). Defaulting it to num_generations
        # is the smallest value that satisfies the check on a single-process
        # setup — matches how GRPOTrainer batches eval internally.
        per_device_eval_batch_size=cfg.num_generations,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        num_train_epochs=cfg.num_train_epochs,
        max_steps=(max_steps if max_steps is not None else -1),
        fp16=use_fp16, bf16=use_bf16,
        # Mid-training eval is disabled for GRPO: TRL's prediction_step
        # calls _generate_and_score_completions, which reshapes rewards
        # as (-1, num_generations). At eval time the effective per-prompt
        # completion count can differ from num_generations, so the view
        # errors with "shape '[-1, G]' is invalid for input of size N".
        # The notebook runs its own before/after eval via evaluate_single_step,
        # so we lose nothing by skipping TRL's eval loop here.
        eval_strategy=eval_strategy, eval_steps=cfg.eval_steps,
        save_strategy=save_strategy, save_steps=cfg.save_steps,
        save_total_limit=cfg.save_total_limit,
        logging_steps=1, report_to="wandb", seed=cfg.seed,
        remove_unused_columns=False,   # CRITICAL: preserves task_id for reward_fns
    )
    return GRPOTrainer(
        model=model, processing_class=tokenizer,
        reward_funcs=list(reward_funcs),
        reward_weights=[1.0, 0.15, 0.05],
        args=args, train_dataset=train_ds, eval_dataset=eval_ds,
        callbacks=[EnvHealthCallback(ENV_CLIENT)],
    )


print("GRPOTrainer factory ready.")


## 12 · Optuna hyperparameter search

Six short trials (30 GRPO steps each, 80-row training subset). Each trial returns the mean **env reward on the held-out val set** — the one metric we actually want to maximize.

- **Resumable**: sqlite storage + `load_if_exists=True`. A dropped Colab session picks up where it left off.
- **Pruned**: `MedianPruner` — kill trials that are trending below the median after 10 steps.
- **Search space** chosen to bracket the GRPO-on-3B defaults reported in the DeepSeek-Math paper.

In [ ]:
import json
from dataclasses import asdict


class CheckpointManager:
    """Find the latest `checkpoint-N/` under `root` for safe resume."""

    def __init__(self, root: Path) -> None:
        self.root = root

    def latest(self) -> str | None:
        if not self.root.exists():
            return None
        ckpts = sorted(
            (d for d in self.root.glob("checkpoint-*") if d.is_dir()),
            key=lambda d: int(d.name.split("-")[-1]),
        )
        return str(ckpts[-1]) if ckpts else None


FINAL_RUN_DIR = OUT_DIR / "final_grpo"
ckpt_mgr = CheckpointManager(FINAL_RUN_DIR)
resume_from = ckpt_mgr.latest()

# Make the final-run cell re-runnable after a kernel restart: if `best_cfg` is
# not in the namespace (Optuna didn't run this session), recover it from the
# persisted optuna_best.json; if that's missing too, fall back to TRAIN
# defaults so the run still launches on the sane baseline hparams.
try:
    best_cfg
except NameError:
    _best_path = OUT_DIR / "optuna_best.json"
    if _best_path.exists():
        _resolved = json.loads(_best_path.read_text())["resolved_config"]
        best_cfg = replace(TRAIN, **{
            k: v for k, v in _resolved.items()
            if k in TrainingConfig.__dataclass_fields__
        })
        print(f"Recovered best_cfg from {_best_path}")
    else:
        best_cfg = TRAIN
        print("No Optuna results found; using TRAIN defaults for best_cfg")

# Stable wandb run id across resumes — same dashboard panel
run_id_file = OUT_DIR / "wandb_final_run_id.txt"
if run_id_file.exists():
    os.environ["WANDB_RUN_ID"] = run_id_file.read_text().strip()
    os.environ["WANDB_RESUME"] = "allow"
else:
    import uuid
    new_id = uuid.uuid4().hex[:8]
    run_id_file.write_text(new_id)
    os.environ["WANDB_RUN_ID"] = new_id

# Master curriculum for the final run — single instance that progresses
# through tiers as the agent demonstrates mastery. Lives for the whole run.
FINAL_CURRICULUM = Curriculum(tasks_dir=_tasks_dir)
# Pre-materialise enough prompts for the full final run. max_steps batches,
# each of size pdtbs × grad_accum prompts, plus a 20% buffer so end-of-epoch
# rounding never exhausts the dataset before max_steps stops training.
_final_n_rows = int(
    PIPE.final_max_steps
    * best_cfg.per_device_train_batch_size
    * best_cfg.gradient_accumulation_steps
    * 1.2
)
FINAL_TRAIN_DS = make_curriculum_dataset(FINAL_CURRICULUM, _final_n_rows)
final_env_r, final_fmt_r, final_len_r = build_reward_funcs(
    ENV_CLIENT, TASK_MAP, FINAL_CURRICULUM,
)

final_model, final_tok = load_policy(MODEL, trainable=True)
final_run = wandb.init(
    project=PIPE.wandb_project, entity=PIPE.wandb_entity,
    name="final-grpo",
    id=os.environ["WANDB_RUN_ID"],
    resume="allow",
    config={**asdict(best_cfg), "phase": "final", "resume_from": resume_from,
            "curriculum": True, "max_steps": PIPE.final_max_steps},
)

final_trainer = build_trainer(
    final_model, final_tok,
    train_ds=FINAL_TRAIN_DS,
    eval_ds=VAL_DS,
    reward_funcs=(final_env_r, final_fmt_r, final_len_r),
    cfg=best_cfg,
    output_dir=str(FINAL_RUN_DIR),
    wandb_run_name=final_run.name,
    use_fp16=RT.use_fp16, use_bf16=RT.use_bf16,
    max_steps=PIPE.final_max_steps,
    save_strategy="steps",
    # Skip TRL's mid-training eval — its GRPO prediction_step reshapes
    # rewards as (-1, num_generations) which blows up when eval generates
    # a different completion count per prompt than training. We do full
    # env-reward eval outside TRL via evaluate_single_step anyway.
    eval_strategy="no",
)

if resume_from:
    print(f"Resuming GRPO from {resume_from}")
final_trainer.train(resume_from_checkpoint=resume_from)

# Log final curriculum stats so judges can see tier progression / mastery counts
final_stats = FINAL_CURRICULUM.get_stats()
print(f"Final curriculum stats: {final_stats}")
wandb.log({f"curriculum/final_{k}": v for k, v in final_stats.items()
           if isinstance(v, (int, float, str))})

# Persist the final LoRA adapter locally before anything else touches VRAM
adapter_local = OUT_DIR / "grpo_adapter"
final_trainer.model.save_pretrained(str(adapter_local))
final_tok.save_pretrained(str(adapter_local))
print(f"Saved GRPO adapter to {adapter_local}")


In [ ]:
# Reclaim VRAM before starting trials. If the user ran the final-run cell
# first (e.g. because optuna_best.json didn't exist on a prior session),
# `final_model` / `final_trainer` / `_probe_model` may still be resident
# on the GPU. Each trial loads a fresh 4-bit Qwen (~6 GB) via
# load_policy, and on a 14.5 GB T4 the second model won't fit alongside
# leftovers, so BitsAndBytes falls back to CPU offload and errors out
# ("Some modules are dispatched on the CPU or the disk").
for _name in ("final_trainer", "final_model", "final_tok", "_probe_model", "_probe_tok", "_baseline_model", "_baseline_tok"):
    if _name in globals():
        try:
            _obj = globals().pop(_name)
            del _obj
        except Exception:
            pass
gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass

STUDY_DB = OUT_DIR / "optuna.db"
STUDY_NAME = "aws-rl-grpo-search"

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=TRAIN.seed),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=10),
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
)

completed = sum(1 for t in study.trials if t.state.name == "COMPLETE")
remaining = max(0, PIPE.n_trials - completed)
print(f"Optuna study '{STUDY_NAME}': {completed} completed, {remaining} remaining.")

if remaining > 0:
    study.optimize(trial_objective, n_trials=remaining)

best_cfg = replace(TRAIN, **{
    k: v for k, v in study.best_params.items() if k in TrainingConfig.__dataclass_fields__
})
(OUT_DIR / "optuna_best.json").write_text(json.dumps({
    "best_value": study.best_value,
    "best_params": study.best_params,
    "resolved_config": asdict(best_cfg),
}, indent=2))

print(f"Best trial env_reward_mean = {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-04-23 15:32:37,465] A new study created in RDB with name: aws-rl-grpo-search
[W 2026-04-23 15:32:37,516] Trial 0 failed with parameters: {'learning_rate': 4.328450221293881e-06, 'beta': 0.09507143064099162, 'num_generations': 4, 'temperature': 0.7468055921327309, 'lora_r': 32, 'lora_alpha_mul': 2} because of the following error: NotImplementedError('Unsloth currently only works on NVIDIA, AMD and Intel GPUs.').
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/pz/c0mcy3kn27l_zxcmb7mbsdww0000gp/T/ipykernel_5236/127989900.py", line 27, in trial_objective
    model, tokenizer = load_policy(MODEL, trainable=True)
                       ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/pz/c0mcy3kn27l_zxcmb7mbsdww0000gp/T/ipykernel_5236/3540064579.py", line 12, in load_policy
    from unsloth import F

Optuna study 'aws-rl-grpo-search': 0 completed, 6 remaining.


NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

In [ ]:
import plotly.io as pio

try:
    fig_history  = optuna.visualization.plot_optimization_history(study)
    fig_parallel = optuna.visualization.plot_parallel_coordinate(study)
    fig_import   = optuna.visualization.plot_param_importances(study)
    for name, fig in (("history", fig_history), ("parallel", fig_parallel), ("importances", fig_import)):
        out_png = OUT_DIR / f"optuna_{name}.png"
        pio.write_image(fig, str(out_png), width=900, height=500)
        print(f"Saved {out_png}")
    # Show inline as well
    fig_history.show(); fig_parallel.show(); fig_import.show()
except Exception as e:
    print(f"Optuna plots skipped: {e}")

[W 2026-04-23 15:32:39,901] There are no complete trials.
[W 2026-04-23 15:32:39,941] Your study does not have any completed trials.
[W 2026-04-23 15:32:39,944] Study instance does not contain completed trials.


Saved /Users/ukpadhy/Personal/aws-rl-env/train/out/optuna_history.png
Saved /Users/ukpadhy/Personal/aws-rl-env/train/out/optuna_parallel.png
Saved /Users/ukpadhy/Personal/aws-rl-env/train/out/optuna_importances.png
Optuna plots skipped: Mime type rendering requires nbformat>=4.2.0 but it is not installed


## 14 · Final GRPO run — best hparams, full data, checkpointed

Uses the Optuna winner on the full 1-epoch training set with periodic checkpoints (`save_steps=25`, `save_total_limit=3`). `CheckpointManager.latest()` auto-detects a prior checkpoint so re-running this cell after a disconnect resumes safely.

Wandb run `final-grpo` gets a stable run id so resumed training stitches into the same panel instead of creating a fresh chart.

## 15 · Multi-step evaluation harness

Training was single-step for TRL compatibility; *evaluation* runs full episodes so we can measure:

- per-tier task success
- hints used per solved task
- recovery rate after chaos injection
- drift repair rate
- steps to solve
- rollback count (safety)
- generalization gap (reserve vs train-held-out)

The pattern mirrors [`scripts/grpo_train.py`](https://github.com/UdayKiranPadhy/aws-rl-env/blob/master/scripts/grpo_train.py)'s `run_single_rollout`, but uses `GrpoPool` for 8-way concurrent rollouts so a 100-task eval finishes in ~7 minutes.

In [ ]:
import asyncio
from collections import defaultdict
from models import AwsRlAction
from client import AwsRlEnv
from scripts.grpo_pool import GrpoPool


SYSTEM_PROMPT = (
    "You are an expert AWS SRE agent. You operate a simulated AWS cloud by "
    "emitting one AWS CLI command at a time. You will see the task description "
    "and the most recent command output, then reply with EXACTLY ONE AWS CLI "
    "command on a single line starting with 'aws '. No explanation, no markdown, "
    "no quotes \u2014 just the command."
)


def build_multi_step_prompt(tokenizer, task: Task,
                             history: list[tuple[str, str]]) -> str:
    """Assemble chat-style prompt including the last few (cmd, output) turns."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"TASK: {task.description}"},
    ]
    for cmd, out in history[-4:]:   # last 4 turns fit in 512 tokens
        messages.append({"role": "assistant", "content": cmd})
        messages.append({"role": "user",      "content": f"OUTPUT:\n{out[:400]}"})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


@dataclass
class EpisodeResult:
    task_id: int
    tier: str
    is_drift: bool
    achieved: bool
    terminal_reward: float
    steps_taken: int
    hints_used: int
    chaos_occurred: bool
    command_failures: int


async def run_episode(env: AwsRlEnv, model, tokenizer,
                      task: Task, drift_ids: set[int],
                      max_steps: int = 15,
                      max_new_tokens: int = 96) -> EpisodeResult:
    """Roll one episode against one env session. Sampling temperature is low
    to reflect deployment behaviour rather than training-time exploration."""
    device = next(model.parameters()).device
    res = await env.reset(task=task)
    history: list[tuple[str, str]] = []
    steps_taken = 0
    command_failures = 0
    terminal_reward = 0.0
    achieved = False

    for _ in range(max_steps):
        steps_taken += 1
        prompt = build_multi_step_prompt(tokenizer, task, history)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.4,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(
            ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
        )
        cmd = extract_aws_command(text)
        res = await env.step(AwsRlAction(command=cmd))
        terminal_reward = float(res.reward)
        obs = res.observation
        if not obs.command_success:
            command_failures += 1
        history.append((cmd, obs.command_output or ""))
        if obs.task_achieved:
            achieved = True
        if res.done:
            break

    # One final /state poll for chaos flag — TrackerState doesn't expose
    # rollback counts, so we skip that metric rather than report zeros.
    try:
        state = await env.state()
        chaos = bool(getattr(state, "chaos_occurred", False))
    except Exception:
        chaos = False

    return EpisodeResult(
        task_id=int(task.task_id),
        tier=task.difficulty.value,
        is_drift=int(task.task_id) in drift_ids,
        achieved=achieved,
        terminal_reward=terminal_reward,
        steps_taken=steps_taken,
        hints_used=int(getattr(res.observation, "hints_used", 0) or 0),
        chaos_occurred=chaos,
        command_failures=command_failures,
    )


print("run_episode defined.")

run_episode defined.


In [ ]:
@dataclass
class RichMetrics:
    """All the metrics the hackathon judges care about."""
    success_by_tier:       dict
    reward_by_tier:        dict
    overall_success_rate:  float
    overall_reward_mean:   float
    hints_per_solved:      float
    recovery_rate:         float
    drift_repair_rate:     float
    steps_to_solve:        float
    destructive_fail_rate: float
    n_episodes:            int

    def as_dict(self) -> dict:
        return asdict(self)


def summarize_episodes(results: list[EpisodeResult]) -> RichMetrics:
    """Aggregate per-tier and overall stats from a list of EpisodeResults.

    Drift detection uses the per-result `is_drift` flag (set from DRIFT_TASK_IDS)
    rather than a tier string — drift tasks live inside the EXPERT tier files.
    """
    by_tier: dict[str, list[EpisodeResult]] = defaultdict(list)
    for r in results:
        by_tier[r.tier].append(r)

    success_by_tier = {tier: sum(r.achieved for r in xs) / max(1, len(xs))
                       for tier, xs in by_tier.items()}
    reward_by_tier  = {tier: (sum(r.terminal_reward for r in xs) / max(1, len(xs)))
                       for tier, xs in by_tier.items()}

    solved = [r for r in results if r.achieved]
    chaos_episodes = [r for r in results if r.chaos_occurred]
    drift_episodes = [r for r in results if r.is_drift]

    return RichMetrics(
        success_by_tier=success_by_tier,
        reward_by_tier=reward_by_tier,
        overall_success_rate=len(solved) / max(1, len(results)),
        overall_reward_mean=sum(r.terminal_reward for r in results) / max(1, len(results)),
        hints_per_solved=(sum(r.hints_used for r in solved) / len(solved)) if solved else 0.0,
        recovery_rate=(sum(r.achieved for r in chaos_episodes) / len(chaos_episodes))
                      if chaos_episodes else 0.0,
        drift_repair_rate=(sum(r.achieved for r in drift_episodes) / len(drift_episodes))
                          if drift_episodes else 0.0,
        steps_to_solve=(sum(r.steps_taken for r in solved) / len(solved)) if solved else 0.0,
        destructive_fail_rate=sum(r.command_failures > 0 for r in results) / max(1, len(results)),
        n_episodes=len(results),
    )


async def evaluate_multi_step(base_url: str, task_ids: list[int],
                              task_map: dict[int, Task],
                              drift_ids: set[int],
                              model, tokenizer,
                              pool_size: int = 8,
                              max_steps: int = 15) -> RichMetrics:
    """Run one episode per task_id across `pool_size` concurrent env sessions."""
    results: list[EpisodeResult] = []
    async with GrpoPool(base_url=base_url, size=pool_size) as pool:
        for start in range(0, len(task_ids), pool_size):
            chunk = task_ids[start:start + pool_size]
            coros = [run_episode(env, model, tokenizer, task_map[tid], drift_ids,
                                  max_steps=max_steps)
                     for env, tid in zip(pool.envs, chunk)]
            chunk_results = await asyncio.gather(*coros, return_exceptions=True)
            for r in chunk_results:
                if isinstance(r, EpisodeResult):
                    results.append(r)
                else:
                    print(f"[eval] episode raised {type(r).__name__}: {r}")
    return summarize_episodes(results)


def select_eval_task_ids(reserve_ds, drift_ids: set[int], cap: int) -> list[int]:
    """Task ids for the multi-step eval: reserve split + every drift task."""
    tids = [int(r["task_id"]) for r in reserve_ds][:cap] if reserve_ds else []
    for did in drift_ids:
        if did not in tids:
            tids.append(did)
    return tids


print("evaluate_multi_step defined.")

evaluate_multi_step defined.


## 16 · Before / after multi-step evaluation

Runs the rich multi-step evaluator once with the **SFT-only** model (baseline) and once with the **final GRPO** model, then prints a delta table and logs every metric to wandb under the `eval/*` and `eval/delta_*` namespaces for judge-facing charts.

In [ ]:
def _flatten_metrics(m: RichMetrics, prefix: str) -> dict:
    out = {}
    for k, v in m.as_dict().items():
        if isinstance(v, dict):
            for tier, val in v.items():
                out[f"{prefix}/{k}/{tier}"] = val
        else:
            out[f"{prefix}/{k}"] = v
    return out


def _delta_metrics(before: RichMetrics, after: RichMetrics) -> dict:
    b, a = before.as_dict(), after.as_dict()
    delta: dict[str, float] = {}
    for k in a:
        if isinstance(a[k], dict):
            for tier, v in a[k].items():
                bv = b.get(k, {}).get(tier, 0.0)
                delta[f"eval/delta_{k}/{tier}"] = v - bv
        elif isinstance(a[k], (int, float)):
            delta[f"eval/delta_{k}"] = a[k] - b[k]
    return delta


EVAL_TASK_IDS = select_eval_task_ids(RESERVE_DS, DRIFT_TASK_IDS, cap=PIPE.eval_reserve_cap)
print(f"Multi-step eval on {len(EVAL_TASK_IDS)} tasks "
      f"({sum(1 for t in EVAL_TASK_IDS if t in DRIFT_TASK_IDS)} drift).")

# --- SFT baseline ---
print("Evaluating SFT baseline (multi-step)...")
sft_model, sft_tok = load_policy(MODEL, trainable=False)
sft_metrics = await evaluate_multi_step(
    ENV_BASE_URL, EVAL_TASK_IDS, TASK_MAP, DRIFT_TASK_IDS,
    sft_model, sft_tok, pool_size=PIPE.env_pool_size,
)
free_model(sft_model); del sft_tok
gc.collect(); torch.cuda.empty_cache()
(OUT_DIR / "baseline_multi_step.json").write_text(json.dumps(sft_metrics.as_dict(), indent=2))

# --- GRPO after ---
print("Evaluating GRPO-trained model (multi-step)...")
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(final_trainer.model)
grpo_metrics = await evaluate_multi_step(
    ENV_BASE_URL, EVAL_TASK_IDS, TASK_MAP, DRIFT_TASK_IDS,
    final_trainer.model, final_tok, pool_size=PIPE.env_pool_size,
)
(OUT_DIR / "grpo_multi_step.json").write_text(json.dumps(grpo_metrics.as_dict(), indent=2))

# --- Deltas + wandb ---
deltas = _delta_metrics(sft_metrics, grpo_metrics)
wandb.log({
    **_flatten_metrics(sft_metrics,  "eval/sft"),
    **_flatten_metrics(grpo_metrics, "eval/grpo"),
    **deltas,
})

# Judge-facing table
def _render_row(name, b, a):
    return f"| {name:<26} | {b:>12.3f} | {a:>12.3f} | {a - b:+.3f} |"

print("\n| Metric                     | SFT baseline | GRPO        | Delta  |")
print("|----------------------------|-------------:|------------:|-------:|")
print(_render_row("overall_success_rate",  sft_metrics.overall_success_rate,  grpo_metrics.overall_success_rate))
print(_render_row("overall_reward_mean",   sft_metrics.overall_reward_mean,   grpo_metrics.overall_reward_mean))
print(_render_row("hints_per_solved",      sft_metrics.hints_per_solved,      grpo_metrics.hints_per_solved))
print(_render_row("recovery_rate",         sft_metrics.recovery_rate,         grpo_metrics.recovery_rate))
print(_render_row("drift_repair_rate",     sft_metrics.drift_repair_rate,     grpo_metrics.drift_repair_rate))
print(_render_row("steps_to_solve",        sft_metrics.steps_to_solve,        grpo_metrics.steps_to_solve))
print(_render_row("destructive_fail_rate", sft_metrics.destructive_fail_rate, grpo_metrics.destructive_fail_rate))

for tier in sorted(set(sft_metrics.success_by_tier) | set(grpo_metrics.success_by_tier)):
    b = sft_metrics.success_by_tier.get(tier, 0.0)
    a = grpo_metrics.success_by_tier.get(tier, 0.0)
    print(_render_row(f"success[{tier}]", b, a))

Multi-step eval on 109 tasks (9 drift).
Evaluating SFT baseline (multi-step)...


NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

## 17 · Qualitative rollouts — one task per tier

A small set of showable transcripts (task, generated command, env output, reward). Judges love seeing actual agent behaviour, not just numbers.

In [ ]:
async def qualitative_rollouts(model, tokenizer, task_map: dict[int, Task],
                                drift_ids: set[int],
                                samples_per_tier: int = 1) -> list[dict]:
    """Pick one task per difficulty tier and print a full rollout transcript."""
    per_tier: dict[str, list[Task]] = defaultdict(list)
    for t in task_map.values():
        per_tier[t.difficulty.value].append(t)
    chosen: list[Task] = []
    for tier in ["warmup", "beginner", "intermediate", "advanced", "expert"]:
        if per_tier.get(tier):
            chosen.extend(per_tier[tier][:samples_per_tier])

    transcripts = []
    async with GrpoPool(base_url=ENV_BASE_URL, size=min(len(chosen), PIPE.env_pool_size)) as pool:
        for env, task in zip(pool.envs, chosen):
            ep = await run_episode(env, model, tokenizer, task, drift_ids, max_steps=8)
            transcripts.append({
                "tier": task.difficulty.value,
                "task_id": int(task.task_id),
                "description": task.description,
                "achieved": ep.achieved,
                "terminal_reward": ep.terminal_reward,
                "steps_taken": ep.steps_taken,
            })
    return transcripts


qual = await qualitative_rollouts(final_trainer.model, final_tok, TASK_MAP, DRIFT_TASK_IDS)
print(json.dumps(qual, indent=2, default=str))
(OUT_DIR / "qualitative_rollouts.json").write_text(json.dumps(qual, indent=2, default=str))

NameError: name 'final_trainer' is not defined

## 18 · Publish the GRPO adapter to a new HF Hub repo

Pushes **only** to `Sizzing/aws-rl-grpo-qwen25coder3b-adapter`. The existing SFT adapter repo `Sizzing/aws-rl-sft-qwen25coder3b-adapter` is never touched — both coexist on Hub so reviewers can load either and compare side by side.

The model card notes the lineage (`base_model = SFT adapter`) so anyone opening the repo on Hub sees immediately that this is a second-stage RL fine-tune.

In [ ]:
from datetime import datetime, timezone


def write_model_card(adapter_dir: Path, model_spec: ModelSpec,
                     cfg: TrainingConfig, grpo: RichMetrics, sft: RichMetrics) -> Path:
    """Emit a README.md for the pushed repo documenting training recipe + metrics."""
    card = f"""---
library_name: peft
base_model: {model_spec.sft_adapter}
pipeline_tag: text-generation
tags: [grpo, lora, aws, reinforcement-learning]
---

# aws-rl-grpo-qwen25coder3b-adapter

GRPO (Group Relative Policy Optimization) LoRA adapter continuing from
[`{model_spec.sft_adapter}`](https://huggingface.co/{model_spec.sft_adapter}).
Trained single-step with live AWS RL env rewards; evaluated multi-step.

## How to load

```python
from unsloth import FastLanguageModel
from peft import PeftModel

base, tok = FastLanguageModel.from_pretrained(
    "{model_spec.base_model}", max_seq_length={model_spec.max_seq_length}, load_in_4bit=True,
)
model = PeftModel.from_pretrained(base, "{model_spec.grpo_adapter}")
FastLanguageModel.for_inference(model)
```

## Training recipe

| Knob                  | Value              |
|-----------------------|--------------------|
| learning_rate         | {cfg.learning_rate:.2e}   |
| beta (KL coef)        | {cfg.beta:.3f}             |
| num_generations (G)   | {cfg.num_generations}                  |
| temperature           | {cfg.temperature:.2f}              |
| lora_r / alpha_mul    | {cfg.lora_r} / {cfg.lora_alpha_mul}                   |
| max_completion_length | {cfg.max_completion_length}                |
| per-device batch      | {cfg.per_device_train_batch_size} x {cfg.gradient_accumulation_steps} accum       |

## Multi-step eval (reserve + drift)

| Metric                | SFT baseline | GRPO   | Delta   |
|-----------------------|-------------:|-------:|--------:|
| overall_success_rate  | {sft.overall_success_rate:.3f}        | {grpo.overall_success_rate:.3f}  | {grpo.overall_success_rate - sft.overall_success_rate:+.3f} |
| overall_reward_mean   | {sft.overall_reward_mean:.3f}        | {grpo.overall_reward_mean:.3f}  | {grpo.overall_reward_mean - sft.overall_reward_mean:+.3f} |
| hints_per_solved      | {sft.hints_per_solved:.3f}        | {grpo.hints_per_solved:.3f}  | {grpo.hints_per_solved - sft.hints_per_solved:+.3f} |
| recovery_rate         | {sft.recovery_rate:.3f}        | {grpo.recovery_rate:.3f}  | {grpo.recovery_rate - sft.recovery_rate:+.3f} |
| drift_repair_rate     | {sft.drift_repair_rate:.3f}        | {grpo.drift_repair_rate:.3f}  | {grpo.drift_repair_rate - sft.drift_repair_rate:+.3f} |
| steps_to_solve        | {sft.steps_to_solve:.3f}        | {grpo.steps_to_solve:.3f}  | {grpo.steps_to_solve - sft.steps_to_solve:+.3f} |

Trained {datetime.now(timezone.utc).isoformat(timespec='minutes')} on {RT.gpu_name}.
"""
    card_path = adapter_dir / "README.md"
    card_path.write_text(card)
    return card_path


# Write card locally, then push both adapter files + card
adapter_local = OUT_DIR / "grpo_adapter"
write_model_card(adapter_local, MODEL, best_cfg, grpo_metrics, sft_metrics)

commit_msg = f"GRPO run {datetime.now(timezone.utc).isoformat(timespec='minutes')}"
final_trainer.model.push_to_hub(
    MODEL.grpo_adapter, private=True, token=HF_TOKEN, commit_message=commit_msg,
)
final_tok.push_to_hub(
    MODEL.grpo_adapter, private=True, token=HF_TOKEN, commit_message=commit_msg,
)
HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=str(adapter_local / "README.md"),
    path_in_repo="README.md",
    repo_id=MODEL.grpo_adapter,
    commit_message=commit_msg,
)
print(f"Pushed to https://huggingface.co/{MODEL.grpo_adapter}")
print(f"SFT adapter at {MODEL.sft_adapter} untouched — both models available on Hub.")

NameError: name 'best_cfg' is not defined

## 19 · Clean shutdown + artifact bundle

Closes wandb, releases the GPU, tars `OUT_DIR` into a single downloadable archive (Colab `files.download`). Nothing else needs to be killed — the env server is hosted externally.

In [ ]:
import tarfile


def bundle_artifacts(out_dir: Path) -> Path:
    """Tar up the run's artifacts for easy download off the VM."""
    archive = out_dir.parent / f"{out_dir.name}.tar.gz"
    keep = ("grpo_adapter", "optuna.db", "optuna_best.json",
            "baseline_single_step.json", "baseline_multi_step.json",
            "grpo_multi_step.json", "qualitative_rollouts.json",
            "wandb_final_run_id.txt", "optuna_history.png",
            "optuna_parallel.png", "optuna_importances.png")
    with tarfile.open(archive, "w:gz") as tar:
        for name in keep:
            p = out_dir / name
            if p.exists():
                tar.add(p, arcname=name)
    return archive


# Release GPU + wandb first
free_model(final_trainer)
del final_trainer, final_model, final_tok
gc.collect(); torch.cuda.empty_cache()

try:
    wandb.finish()
except Exception as e:
    print(f"wandb.finish warning: {e}")

archive = bundle_artifacts(OUT_DIR)
print(f"\nBundle: {archive}")
print(f"\nHF Hub:\n  SFT:  https://huggingface.co/{MODEL.sft_adapter}\n  GRPO: https://huggingface.co/{MODEL.grpo_adapter}")

if IS_COLAB:
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as e:
        print(f"Colab auto-download skipped: {e}\nDownload manually from {archive}")

NameError: name 'final_trainer' is not defined